In [ ]:
# Nhập các thư viện cần thiết
import pandas as pd
from fuzzywuzzy import process, fuzz

In [ ]:
# Tải dữ liệu thô từ 2 nguồn EduRank và OpenAlex
df_edu = pd.read_csv('edurank_ai_rankings.csv')
df_oa = pd.read_csv('openalex_ai_vietnam.csv')

In [ ]:
# Tiền xử lý: Tách và nhân bản các bài báo có nhiều trường cùng hợp tác (cách nhau bởi dấu |)
df_oa['University_Name'] = df_oa['University_Name'].str.split(' \\| ')
df_oa = df_oa.explode('University_Name')
df_oa['University_Name'] = df_oa['University_Name'].str.strip()

In [ ]:
# Tính tổng: Đếm số lượng bài báo và tổng lượt trích dẫn cho từng trường
df_oa_agg = df_oa.groupby('University_Name').agg(
    OA_Papers=('Paper_ID', 'count'),
    OA_Citations=('Citations', 'sum')
).reset_index()

In [ ]:
# Chuẩn hoá: Dùng fuzz.token_sort_ratio để tránh ảo giác ghép nhầm từ khoá chung chung (như 'University')
edu_names = df_edu['University_Name'].dropna().tolist()

def match_name(name):
    if pd.isna(name) or name == '': 
        return None
    match, score = process.extractOne(str(name), edu_names, scorer=fuzz.token_sort_ratio)
    return match if score >= 85 else None

df_oa_agg['Matched_Name'] = df_oa_agg['University_Name'].apply(match_name)

In [ ]:
# Gộp tổng lại một lần nữa cho những trường OpenAlex có cùng Matched_Name
df_oa_final = df_oa_agg.dropna(subset=['Matched_Name']).groupby('Matched_Name').agg(
    OA_Papers=('OA_Papers', 'sum'),
    OA_Citations=('OA_Citations', 'sum')
).reset_index()

In [ ]:
# Hợp nhất (Merge): Ghép dữ liệu OpenAlex vào EduRank (tránh bị trùng lặp dòng), xuất ra file CSV hoàn chỉnh
df_merged = pd.merge(df_edu, df_oa_final, left_on='University_Name', right_on='Matched_Name', how='left')
df_merged = df_merged.drop(columns=['Matched_Name'])
df_merged.to_csv('merged_data.csv', index=False)
df_merged.head()